# Salma — NeuroMatch 2026 · Posterior Motives

**Your slide's questions:**

- Examine prior learning **within one coherence level**.
- Compare learning across prior SDs: **10°, 20°, 40°, 80°**.
- Quantify across all subjects for each prior context.
- Plot learned prior width / confidence across prior widths.
- Check block/plot alignment (a possible block shift was discussed).

This notebook is runnable top-to-bottom and uses **HB-Rachel** (our hierarchical Bayesian observer) and the **Switch** (paper's switching observer). Edit `SUBJECT` / filters and re-run any cell. See the API guide cell for how to make your own calls.

In [ ]:
# ============================================================
#  SETUP  —  works in Google Colab or on a local checkout
# ============================================================
import os, sys, subprocess

REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
BRANCH   = "model-verification"     # branch that carries the fitted models + API

def _find_root():
    """If we're already inside a checkout, use it (no clone needed)."""
    here = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(here, "observers", "api.py")):
            return here
        here = os.path.dirname(here)
    return None

ROOT = _find_root()
if ROOT is None:
    # Colab path: shallow-clone the repo, then point at the hierarchical/ package.
    dest = "/content/NeuroMatch_2026_Behaviour" if os.path.isdir("/content") \
           else os.path.abspath("NeuroMatch_2026_Behaviour")
    if not os.path.isdir(dest):
        # PUBLIC repo: this just works. PRIVATE repo: replace REPO_URL with
        #   https://<TOKEN>@github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                        REPO_URL, dest], check=True)
    ROOT = os.path.join(dest, "hierarchical")

sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from observers import api

print("repo root :", ROOT)
print("models    :", api.list_models())
print("data      :", "data/data01_direction4priors.csv  (12 subjects)")

# figures for this notebook go in a dedicated subfolder beside it
FIG_DIR = os.path.join(ROOT, "experiments", "salma", "01_slide_notebook", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("figures  :", FIG_DIR)

## How to use the model API (read me — also for an LLM assistant)

Everything below comes from the **fitted models**, reached through one module:
`observers.api`. You never touch raw model math — you call functions.

### Where the data lives (relative to the repo's `hierarchical/` folder)
| what | path |
|---|---|
| trial-level data (all 12 subjects) | `data/data01_direction4priors.csv` |
| point fits (per model, per subject) | `results/fits/comparison/<model>/subject<N>.json` |
| cross-validation records | `results/fits/comparison_cv/<model>/subject<N>_cv.json` |
| project abstract | `docs/project_abstract.md` |

### Model keys
`'switch'` (paper's Switching observer, 9 params, non-learning) · `'hb_rachel'`
(our hierarchical Bayesian observer, 7 params, **learns** prior width online) ·
also available: `'hb_adaptive'`, `'hb_salma'`, `'recombined'`, `'basic_bayes'`.

### The API — five kinds of call
```python
from observers import api

# LOAD --------------------------------------------------------------
api.load_subject(2)                 # -> DataFrame: motion_direction, motion_coherence,
                                    #    prior_std, estimate_dir  (one subject's trials)
api.load_fitted('hb_rachel', 2)     # -> (observer, record) with fitted params
api.observed_distribution(2, direction=335, coherence=0.06, prior_std=80)
                                    # -> empirical response histogram (density over 1..360)

# INSPECT -----------------------------------------------------------
api.list_models(); api.model_info() # what exists, params, colors
api.fitted_subjects('hb_rachel')    # which subjects are fit

# PREDICT -----------------------------------------------------------
api.predict('hb_rachel', 2)         # -> (n_trials, 360) predicted distribution per trial
api.belief_trajectory('hb_rachel', 2)
                                    # -> DataFrame trial, believed_sd  (the LEARNED prior width)

# FIT / SIMULATE ----------------------------------------------------
api.fit_model('hb_rachel', 2)       # refit from scratch (slow)
api.simulate('hb_rachel', 2, seed=0)# generative: synthetic responses from the fitted model

# EVALUATE ----------------------------------------------------------
api.results_table()                 # -> tidy DataFrame: model,label,subject,k,nll,aic,bic,cv_nll
api.trial_logliks('hb_rachel', 2)   # -> per-trial log-likelihood (slice it however you like)
api.bias_variability(2)             # -> per-condition estimation bias + circular SD (Fig-3 core)
```

### Custom calls (when a helper doesn't exist)
The API returns raw numbers; **you** aggregate/plot. To reach the model object
directly:
```python
from observers.comparison.registry import build_registry, load_subject
spec = build_registry(['hb_rachel'])['hb_rachel']   # a ModelSpec
obs, rec = api.load_fitted('hb_rachel', 2)
dists = spec.predict_distributions(obs, load_subject(2))  # (n_trials, 360)
```
Condition labels for any trial-level array come from `api.load_subject(s)` —
they're **row-aligned** with `predict`, `trial_logliks`, and `belief_trajectory`.

## 1 · Learned belief per trial (the shared building block)

`belief_trajectory('hb_rachel', s)` replays the fitted observer over subject *s*'s real trials and returns the **learned prior width** (`believed_sd` = the SD implied by the believed concentration κ) on every trial. Joining it with the condition labels gives us learning by coherence × prior width. (This cell is shared with Romi and Valeria.)

In [ ]:
# ---- shared helper: learned belief joined with per-trial conditions ----
# belief_trajectory() replays the FITTED observer over the subject's real trials
# and returns the learned prior width (believed_sd) per trial. We join it with
# the condition labels (coherence, prior_std), which are ROW-ALIGNED with it.
def belief_with_conditions(key, subject):
    bt = api.belief_trajectory(key, subject)          # trial, believed_sd
    d  = api.load_subject(subject).reset_index(drop=True)
    out = bt.copy()
    out['coherence'] = d['motion_coherence'].values
    out['prior_std'] = d['prior_std'].values
    out['subject']   = subject
    return out

def all_beliefs(key, subjects=range(1,13)):
    return pd.concat([belief_with_conditions(key, s) for s in subjects],
                     ignore_index=True)

allb = all_beliefs('hb_rachel')
print('rows:', len(allb), '| columns:', list(allb.columns))
print(allb.head(3).to_string(index=False))

## 2 · Prior learning within one coherence level, across prior SDs

Fix coherence (the task specifies one level) and ask: does the learned prior width track the true block width 10 → 20 → 40 → 80? We show it at each coherence so you can pick the one for your slide.

In [ ]:
COH = 0.06     # <- edit: 0.06, 0.12, or 0.24 (the one-coherence-level focus)
sub = allb[np.isclose(allb.coherence, COH)]
tab = (sub.groupby(['prior_std','subject']).believed_sd.mean()
          .groupby('prior_std').agg(['mean','std','count']).round(2))
print(f'Learned prior width (believed_sd) at coherence={COH}, across prior SDs:')
print(tab.to_string())
print('\nTrue block SDs are 10/20/40/80 -> learned width should rise monotonically.')

## 3 · Plot: learned width across prior widths (all coherences)

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.6))
colors = {0.06:'#8ecae6', 0.12:'#219ebc', 0.24:'#023047'}
for coh in [0.06, 0.12, 0.24]:
    sub = allb[np.isclose(allb.coherence, coh)]
    m = sub.groupby('prior_std').believed_sd.mean()
    ax.plot(m.index, m.values, 'o-', color=colors[coh], label=f'coh={coh}')
ax.plot([10,20,40,80],[10,20,40,80], 'k:', alpha=0.5, label='true block SD')
ax.set_xlabel('nominal prior SD (deg)'); ax.set_ylabel('learned believed prior SD (deg)')
ax.set_title('HB-Rachel learns block-specific prior width\n(nearly identical across coherence)')
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'salma_learned_width.png'), dpi=130); plt.show()

## 4 · ANOVA — believed_sd ~ coherence + prior_std

In [ ]:
# ---- shared helper: two-way ANOVA believed_sd ~ coherence + prior_std ----
# Uses PER-SUBJECT cell means (not raw trials) to avoid pseudoreplication.
import statsmodels.api as sm
import statsmodels.formula.api as smf
def anova_believed_sd(df):
    cell = df.groupby(['subject','coherence','prior_std']).believed_sd.mean().reset_index()
    model = smf.ols('believed_sd ~ C(coherence) + C(prior_std)', data=cell).fit()
    aov = sm.stats.anova_lm(model, typ=2)
    aov['eta_sq'] = aov['sum_sq'] / aov['sum_sq'].sum()
    return aov[['sum_sq','F','PR(>F)','eta_sq']].round(4)

aov = anova_believed_sd(allb)
print('Two-way ANOVA on learned prior width (per-subject cell means):')
print(aov.to_string())
print('\nExpectation: prior_std dominates (large F, large eta_sq); coherence ~0,')
print('because coherence enters only the sensory likelihood, never the width update.')

## 5 · Block-alignment check (the discussed block shift)

A learning model updates its belief *after* feedback, so the belief on trial *t* reflects blocks up to *t−1*. If plots look shifted by one block, this off-by-one (belief lags the block boundary by design) is usually why. This cell overlays the believed width on the true block width for one subject so you can see the alignment directly.

In [ ]:
S = 2   # example subject
b = belief_with_conditions('hb_rachel', S)
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(b.trial, b.believed_sd, color='#edae49', lw=1.2, label='believed prior SD (learned)')
ax.step(b.trial, b.prior_std, where='post', color='k', alpha=0.5, lw=1, label='true block SD')
ax.set_xlabel('trial'); ax.set_ylabel('prior SD (deg)')
ax.set_title(f'subject {S}: learned width tracks block structure (belief lags boundary by ~1 block by design)')
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'salma_block_alignment.png'), dpi=130); plt.show()